# Population growth and urban structure

In [ ]:
import geopandas as gpd
import utca
import importlib
importlib.reload(utca)
from clustergram import Clustergram
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

import matplotlib as mpl

mpl.rcParams.update({
    # Font
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,

    # Lines and markers
    "lines.linewidth": 1.2,
    "lines.markersize": 4,

    # Axes
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,

    # Layout
    "figure.constrained_layout.use": True,
})

cm = 1 / 2.54

In [ ]:
ksh_hnt = pd.read_csv('data/hnt_2025.csv', usecols=['cityname', 'jogallas', 'megye', 'nepesseg'])
pop = utca.load_population()
pop = pop[pop[2011]>10000]

In [ ]:
pop = pd.merge(pop, ksh_hnt, on='cityname')
pop[2025] = pop['nepesseg']
pop.set_index('cityname', inplace=True)
df = pop.drop(labels=['jogallas', 'megye', 'nepesseg'], axis=1)
years = df.columns
X = df.to_numpy()
data = X / X.max(axis=1, keepdims=True)


In [ ]:
cgram = Clustergram(range(1, 10), n_init=100, random_state=1337)
cgram.fit(data)
cgram.plot()

In [ ]:
num_clusters = 4
labels = cgram.labels[num_clusters]
names = [
    "stagnation",
    "boom town",
    "small growth",
    "big growth",
]

fig, axs = plt.subplots(2,2, figsize=(14*cm, 10*cm), sharex=True, sharey=True)

for cluster in range(num_clusters):
    #plt.figure(figsize=(4, 3))
    ax = axs[cluster//2, cluster%2]
    ax.set_title(names[cluster])
    ax.set_xlabel("Time")
    ax.set_ylabel("Population")
    
    # Select time series belonging to the current cluster
    cluster_data = data[labels == cluster]
    
    # Plot all time series in the current cluster
    for series in cluster_data:
        ax.plot(years, series, alpha=0.3, c='0.7')
    ax.plot(years, cluster_data.mean(axis=0))
    
plt.show()
#fig.savefig("output/figs_maj10/pop_clusters.pdf")

In [ ]:
df['labels'] = labels.to_list()

Read graph stats data: can be generated with `calculate_all_graph_stats.py`

In [ ]:
all_graph_stats = pd.read_csv("output/all_graph_stats_maj8.csv")
all_graph_stats.fillna(0, inplace=True)
all_graph_stats['XYT'] = all_graph_stats['X'] + all_graph_stats['Y'] + all_graph_stats['T']
all_graph_stats['X^'] = all_graph_stats['X'] / all_graph_stats['XYT']
all_graph_stats['Y^'] = all_graph_stats['Y'] / all_graph_stats['XYT']
all_graph_stats['T^'] = all_graph_stats['T'] / all_graph_stats['XYT']

In [ ]:
final = all_graph_stats.join(df[['labels', 2025]], on='cityname', how='inner')


In [ ]:
final

In [ ]:
final['cluster'] = final['labels'].apply(lambda x: names[x])
final = final.rename(columns={2025: 'population'})

In [ ]:
pd.options.plotting.backend = "plotly"
final.plot.scatter(x='n', y='v', color='cluster', hover_data=['cityname'], size='population')

In [ ]:
palette_dict = dict(zip(names, sns.color_palette("tab10", len(names))))

In [ ]:
fig, ax = plt.subplots(figsize=(10*cm, 6*cm))
sns.scatterplot(data=final, x='n', y='v', hue='cluster', ax=ax, palette=palette_dict)
ax.legend(bbox_to_anchor=(1.05, 0.7), loc='upper left', title='Cluster')
ax.set_xlabel("$\\overline{n}$")
ax.set_ylabel("$\\overline{v}$")#, rotation=0)
#fig.savefig("output/figs_jan2/pop_scatter.pdf")

In [ ]:
numeric = final.drop(columns=['cityname', 'labels'])


In [ ]:
numeric.groupby('cluster').mean()

In [ ]:
numeric.groupby('cluster').std()

In [ ]:
numeric = numeric.drop(columns=['N_F', "N_V", "T", 'X', 'Y', 'other', 'XYT', 'population'])

# Calculate mean and std for each cluster
grouped_mean = numeric.groupby('cluster').mean()
grouped_std = numeric.groupby('cluster').std()

# Create LaTeX table
latex_table = "\\begin{table}[h]\n\\centering\n\\begin{tabular}{l"
latex_table += "c" * len(grouped_mean.index) + "}\n\\hline\n"

# Header row
latex_table += "Variable"
for cluster_name in grouped_mean.index:
    latex_table += f" & {cluster_name}"
latex_table += " \\\\\n\\hline\n"

# Data rows
for col in grouped_mean.columns:
    latex_table += col
    for cluster_name in grouped_mean.index:
        mean_val = grouped_mean.loc[cluster_name, col]
        std_val = grouped_std.loc[cluster_name, col]
        latex_table += f" & ${mean_val:.2f} \\pm {std_val:.2f}$"
    latex_table += " \\\\\n"

latex_table += "\\hline\n\\end{tabular}\n\\caption{Cluster statistics}\n\\end{table}"

print(latex_table)

# hungary map

In [ ]:
admin_centres = pd.read_csv("data/admin_centres.csv")

In [ ]:
map_df = pd.merge(admin_centres, final, left_on='name', right_on='cityname')

In [ ]:
fig, ax = plt.subplots(figsize=(10*cm, 6*cm))
path = 'data/hungary_shp/admin2.shp'
gdf = gpd.read_file(path)
gdf = gdf.to_crs(epsg=4326)
gdf.iloc[[0]].boundary.plot(color="0.3", ax=ax, zorder=-1)
#map_df.plot.scatter(x='lon', y='lat', c='cluster', cmap=cmap, ax=ax, zorder=2)
sns.scatterplot(data=map_df, x='lon', y='lat', hue='cluster', ax=ax, palette=palette_dict)
plt.show()

In [ ]:
# Create a figure with two subplots and a common legend
fig, axs = plt.subplots(1, 2, figsize=(14*cm, 6*cm))

# First scatter plot: n vs v
sns.scatterplot(data=final, x='n', y='v', hue='cluster', ax=axs[0], palette=palette_dict, legend=False)
axs[0].set_xlabel("$\\overline{n}^*$")
axs[0].set_ylabel("$\\overline{v}^*$")
#axs[0].set_title("Network metrics")

# Second scatter plot: lon vs lat with boundary
path = 'data/hungary_shp/admin2.shp'
gdf = gpd.read_file(path)
gdf = gdf.to_crs(epsg=4326)
gdf.iloc[[0]].boundary.plot(color="0.3", ax=axs[1], zorder=-1)
sns.scatterplot(data=map_df, x='lon', y='lat', hue='cluster', ax=axs[1], palette=palette_dict, legend=False)
axs[1].set_xlabel("Longitude")
axs[1].set_ylabel("Latitude")

# Despine the right plot completely
#sns.despine(ax=axs[1], left=True, bottom=True, right=True, top=True)
#axs[1].set_axis_off()
sns.despine(ax=axs[1], trim=True)


# Create a common legend from the palette
import matplotlib.patches as mpatches
handles = [mpatches.Patch(facecolor=palette_dict[name], label=name) for name in names]
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.9), ncol=len(names), title='Cluster')

plt.tight_layout()
plt.show()

In [ ]:
#fig.savefig("output/figs_maj10/pop_map.pdf")

In [ ]:
map_gdf = gpd.GeoDataFrame(
    map_df.copy(),
    geometry=gpd.points_from_xy(map_df["lon"], map_df["lat"]),
    crs="EPSG:4326"
)
cmap = sns.mpl_palette('tab10', 4, as_cmap=True)
utca.basic_visu(map_gdf, column='cluster', cmap=cmap)